# CR2032 Publication Quickstart

This notebook demonstrates the smallest useful BattINFO publication flow:

1. create a `CellType -> CellInstance -> Test -> Dataset` chain
2. publish `battinfo.publish.jsonld`
3. reload that JSON-LD back into BattINFO objects


In [ ]:
from pathlib import Path
from pprint import pprint

from battinfo import CellInstance, CellType, Dataset, Test, load_publication, publish

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "assets").exists() and (REPO_ROOT.parent / "assets").exists():
    REPO_ROOT = REPO_ROOT.parent

DATASET_DIR = REPO_ROOT / ".battinfo" / "example-datasets" / "energizer-cr2032-202602-dtjrga"

print("Repo root:", REPO_ROOT)
print("Dataset dir:", DATASET_DIR)


## Build The Core Object Chain

In [ ]:
cell_type = CellType(
    manufacturer="Energizer",
    model="CR2032",
    format="coin",
    chemistry="Li-primary",
    size_code="CR2032",
    nominal_properties={"nominal_voltage": {"value": 3.0, "unit": "V"}},
)

cell = CellInstance(
    cell_type=cell_type,
    serial_number="energizer-cr2032-202602-dtjrga",
)

test = Test(
    cell=cell,
    kind="capacity_check",
    protocol="constant current discharging",
    instrument="short Landt cycler",
    status="completed",
)

dataset = Dataset(
    path=str(DATASET_DIR),
    cell=cell,
    test=test,
    name="Energizer CR2032 dataset",
    description="One constant-current discharge run for a CR2032 cell.",
)

pprint({
    "cell_type": cell_type.model_dump(exclude_none=True),
    "cell_instance": cell.model_dump(exclude_none=True),
    "test": test.model_dump(exclude_none=True),
    "dataset": dataset.model_dump(exclude_none=True),
})


## Publish JSON-LD

`publish(...)` writes `battinfo.publish.jsonld` into the dataset directory.

In [ ]:
report = publish(
    cell_type=cell_type,
    cell_instance=cell,
    test=test,
    dataset=dataset,
)

pprint(report)


## Reload The Publication Artifact

In [ ]:
publish_path = DATASET_DIR / "battinfo.publish.jsonld"
bundle = load_publication(publish_path)

summary = {
    "publish_path": str(publish_path),
    "cell_type": {
        "id": bundle.cell_type.id,
        "name": bundle.cell_type.name,
        "chemistry": bundle.cell_type.chemistry,
        "cell_specification_id": bundle.cell_type.cell_specification_id,
    },
    "cell_instance": {
        "id": bundle.cell_instance.id,
        "serial_number": bundle.cell_instance.serial_number,
    },
    "test": {
        "id": bundle.test.id,
        "kind": bundle.test.test_kind,
        "protocol": bundle.test.protocol.name,
        "instrument": bundle.test.instrument,
    },
    "dataset": {
        "id": bundle.dataset.id,
        "access_url": bundle.dataset.access_url,
    },
}

pprint(summary)


## Inspect The Published Graph

In [ ]:
import json

payload = json.loads(publish_path.read_text(encoding="utf-8"))
pprint([
    {"id": node.get("@id"), "type": node.get("@type")}
    for node in payload.get("@graph", [])
    if isinstance(node, dict)
])
